**Imports**

In [26]:
import os 
from dotenv import load_dotenv
import json
import requests
from openai import OpenAI
import gradio as gr

**Load API keys and profile data from environment variables**

In [10]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY","")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY environment variable not set")
else:
    print("Retrieved Gemini API key from environment variable")

REBRICKABLE_API_KEY = os.getenv("REBRICKABLE_API_KEY","")
if not REBRICKABLE_API_KEY:
    raise ValueError("REBRICKABLE_API_KEY environment variable not set")
else:
    print("Retrieved Rebrickable API key from environment variable")
    
REBRICKABLE_USERNAME = os.getenv("REBRICKABLE_USERNAME","")
if not REBRICKABLE_USERNAME:
    raise ValueError("REBRICKABLE_USERNAME environment variable not set")
else:
    print("Retrieved Rebrickable username from environment variable")
    
REBRICKABLE_PASSWORD = os.getenv("REBRICKABLE_PASSWORD","")
if not REBRICKABLE_PASSWORD:
    raise ValueError("REBRICKABLE_PASSWORD environment variable not set")
else:
    print("Retrieved Rebrickable password from environment variable")



Retrieved Gemini API key from environment variable
Retrieved Rebrickable API key from environment variable
Retrieved Rebrickable username from environment variable
Retrieved Rebrickable password from environment variable


**Set up Gemini**

In [11]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-2.5-flash-lite"

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)


**Get Rebrickable User Token**

In [12]:
REBRICKABLE_LEGO_BASE_URL = "https://rebrickable.com/api/v3/lego/"

def get_user_token():
    url = "https://rebrickable.com/api/v3/users/_token/"
    headers = {'Authorization': f'key {REBRICKABLE_API_KEY}'}
    payload = {
        'username': REBRICKABLE_USERNAME,
        'password': REBRICKABLE_PASSWORD
    }
    response = requests.post(url, headers=headers, data=payload)
    if response.status_code == 200:
        return response.json()["user_token"]
    else:
        raise ValueError(f"Failed to get user token: {response.status_code} {response.text}")

In [13]:
user_token = get_user_token()
REBRICKABLE_USER_BASE_URL = f"https://rebrickable.com/api/v3/users/{user_token}/"
print(f"Retrieved user token: {user_token}")

Retrieved user token: e0646b4ab11cca2eb77dbd807170758f4d325f81b6b5066b2e4aca65119fbb1b


In [30]:
def get_user_sets(min_year=None, max_year=None, min_parts=None, max_parts=None):
    print(f"Getting user sets with filters: min_year={min_year}, max_year={max_year}, min_parts={min_parts}, max_parts={max_parts}")
    url = REBRICKABLE_USER_BASE_URL + "sets/"
    headers = {'Authorization': f'key {REBRICKABLE_API_KEY}'}
    payload = {
        'min_year': min_year,
        'max_year': max_year,
        'min_parts': min_parts,
        'max_parts': max_parts
    }
    sets = []
    
    while url:
        response = requests.get(url, headers=headers, params=payload)
        if response.status_code == 200:
            data = response.json()
            sets.extend(data["results"])
            url = data.get("next")
        else:
            raise ValueError(f"Failed to get user sets: {response.status_code} {response.text}")
    return sets

In [48]:
setlist = get_user_sets()
print(f"Retrieved {len(setlist)} sets from user's collection")
print(f"First set: {setlist[0]['set']['name']}")

Getting user sets with filters: min_year=None, max_year=None, min_parts=None, max_parts=None
Retrieved 127 sets from user's collection
First set: Mercedes-Benz Arocs 3245


Tool call: get_user_sets with arguments {}
Getting user sets with filters: min_year=None, max_year=None, min_parts=None, max_parts=None


In [ ]:
print(setlist)


[{'list_id': 220187, 'quantity': 1, 'include_spares': True, 'set': {'set_num': '42043-1', 'name': 'Mercedes-Benz Arocs 3245', 'year': 2015, 'theme_id': 1, 'num_parts': 2793, 'set_img_url': 'https://cdn.rebrickable.com/media/sets/42043-1/6926.jpg', 'set_url': 'https://rebrickable.com/sets/42043-1/mercedes-benz-arocs-3245/', 'last_modified_dt': '2020-09-05T15:44:33.349224Z'}}, {'list_id': 220187, 'quantity': 1, 'include_spares': True, 'set': {'set_num': '42056-1', 'name': 'Porsche 911 GT3 RS', 'year': 2016, 'theme_id': 1, 'num_parts': 2704, 'set_img_url': 'https://cdn.rebrickable.com/media/sets/42056-1/3156.jpg', 'set_url': 'https://rebrickable.com/sets/42056-1/porsche-911-gt3-rs/', 'last_modified_dt': '2019-08-11T19:44:14.683445Z'}}, {'list_id': 220187, 'quantity': 1, 'include_spares': True, 'set': {'set_num': '42055-1', 'name': 'Bucket Wheel Excavator', 'year': 2016, 'theme_id': 1, 'num_parts': 3929, 'set_img_url': 'https://cdn.rebrickable.com/media/sets/42055-1/6538.jpg', 'set_url': '

### Lego Assistant

**System prompt**

In [32]:
system_prompt = """
    You are a helpful assistant that helps users find out which LEGO sets and parts they have in their collection. When the user asks
    about their collection, you can call the get_user_sets function to retrieve the list of sets they own. 
    You can also filter the sets by year and number of parts. You should be able to answer questions like:
    - How many sets do I have?
    - What is the largest set I have?
    - Do I have any sets from 2020?
"""

**Get sets tool**

In [42]:
get_user_sets_tool = {
    "name": "get_user_sets",
    "description": "Get a list of LEGO sets in the user's collection",
    "parameters": {
        "type": "object",
        "properties": {
            "min_year": {
                "type": "integer",
                "description": "Minimum year of the sets to retrieve"
            },
            "max_year": {
                "type": "integer",
                "description": "Maximum year of the sets to retrieve"
            },
            "min_parts": {
                "type": "integer",
                "description": "Minimum number of parts in the sets to retrieve"
            },
            "max_parts": {
                "type": "integer",
                "description": "Maximum number of parts in the sets to retrieve"
            }
        },
        "required": [],
        "additionalProperties": False
    }
}

In [43]:
tools = [{"type": "function", "function": get_user_sets_tool}]
tools

[{'type': 'function',
  'function': {'name': 'get_user_sets',
   'description': "Get a list of LEGO sets in the user's collection",
   'parameters': {'type': 'object',
    'properties': {'min_year': {'type': 'integer',
      'description': 'Minimum year of the sets to retrieve'},
     'max_year': {'type': 'integer',
      'description': 'Maximum year of the sets to retrieve'},
     'min_parts': {'type': 'integer',
      'description': 'Minimum number of parts in the sets to retrieve'},
     'max_parts': {'type': 'integer',
      'description': 'Maximum number of parts in the sets to retrieve'}},
    'required': [],
    'additionalProperties': False}}}]

In [44]:
tools_map = {
    "get_user_sets": get_user_sets
}

In [45]:
def handle_tool_calls(message):
    response = []
    for tool_call in message.tool_calls:
        print(f"Tool call: {tool_call.function.name} with arguments {tool_call.function.arguments}", flush=True)
        function_name = tool_call.function.name
        if function_name in tools_map:
            function = tools_map[function_name]
            arguments = json.loads(tool_call.function.arguments)
            result = function(**arguments)
            response.append({
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            })
    return response

**Chat function**


In [46]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        response_message = response.choices[0].message
        tool_output = handle_tool_calls(response_message)
        messages.append(response_message)
        messages.extend(tool_output)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()